<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/04-pythonic-tools/03_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 4: Solutions

Try the exercises first. The notes after each solution explain the parts that are easy to get subtly wrong with iterators, decorators and context managers, which is where most of the learning in this module is.

In [1]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import copy
import math
import inspect
import itertools
import time


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, float) or isinstance(b, float):
        if not isinstance(a, (int, float)) or not isinstance(b, (int, float)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=70):
    text = repr(x)
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def _first(iterator, limit=10):
    """Take at most `limit` items, so a buggy infinite generator fails instead of hanging."""
    return list(itertools.islice(iterator, limit))


def check_take(f):
    def stops_early():
        it = iter([1, 2, 3, 4])
        f(2, it)
        return next(it)

    _scenarios("take", [
        ("first 3 items of an infinite iterator", lambda: f(3, itertools.count()), [0, 1, 2],
         "you can't call list() on an infinite iterator. Stop after n items"),
        ("asking for more than there is", lambda: f(5, [1, 2]), [1, 2], None),
        ("n = 0", lambda: f(0, [1, 2]), [], None),
        ("doesn't consume more than n items", stops_early, 3,
         "after taking 2 items, the next item in the iterator should still be 3"),
        ("returns a list", lambda: type(f(2, "abc")).__name__, "list", None),
    ])


def check_running_total(f):
    _scenarios("running_total", [
        ("is a generator", lambda: inspect.isgenerator(f([1])), True, "use yield"),
        ("basic", lambda: list(f([1, 2, 3])), [1, 3, 6], None),
        ("floats", lambda: list(f([0.5, 0.25])), [0.5, 0.75], None),
        ("empty input", lambda: list(f([])), [], None),
        ("works on an infinite iterator", lambda: list(itertools.islice(f(itertools.count(1)), 4)), [1, 3, 6, 10],
         "don't convert the input to a list; loop over it directly"),
    ])


def check_count_calls(f):
    def counts():
        @f
        def add(a, b):
            return a + b
        add(1, 2)
        add(3, 4)
        add(5, b=6)
        return add.calls

    def result():
        @f
        def add(a, b):
            return a + b
        return add(2, b=3)

    def keeps_metadata():
        @f
        def scale(x):
            """Multiply by two."""
            return 2 * x
        return (scale.__name__, scale.__doc__)

    def separate():
        @f
        def a():
            pass

        @f
        def b():
            pass
        a(); a(); b()
        return (a.calls, b.calls)

    _scenarios("count_calls", [
        ("returns the original result", result, 5, "return whatever the wrapped function returns"),
        ("counts calls", counts, 3, "store the count as an attribute on the wrapper: wrapper.calls"),
        ("starts at 0", lambda: f(lambda: None).calls, 0, None),
        ("keeps the name and docstring", keeps_metadata, ("scale", "Multiply by two."), "use @functools.wraps(func)"),
        ("each decorated function has its own count", separate, (2, 1), None),
    ])


def check_Timer(C):
    def measures():
        with C() as t:
            time.sleep(0.02)
        return t.elapsed >= 0.019

    def enter_returns_self():
        timer = C()
        with timer as t:
            pass
        return t is timer

    def survives_error():
        t = C()
        try:
            with t:
                raise KeyError("boom")
        except KeyError:
            pass
        return t.elapsed is not None

    def propagates():
        with C():
            raise KeyError("boom")

    _scenarios("Timer", [
        ("elapsed is None before use", lambda: C().elapsed, None, None),
        ("`with ... as t` gives the Timer itself", enter_returns_self, True, "__enter__ should return self"),
        ("measures elapsed time in seconds", measures, True, "use time.perf_counter() in __enter__ and __exit__"),
        ("elapsed is set even if the block raises", survives_error, True, None),
        ("exceptions are not swallowed", propagates, Raises(KeyError), "__exit__ should return False"),
    ])


def check_batched(f):
    _scenarios("batched", [
        ("is a generator", lambda: inspect.isgenerator(f([1], 1)), True, "use yield"),
        ("basic", lambda: _first(f([1, 2, 3, 4, 5], 2)), [[1, 2], [3, 4], [5]],
         "if the first batch repeats forever, call iter() on the input once and keep using that iterator"),
        ("works on a generator", lambda: _first(f((x for x in range(5)), 2)), [[0, 1], [2, 3], [4]],
         "generators have no len() and can't be sliced. Collect items one at a time"),
        ("lazy on an infinite iterator", lambda: next(f(itertools.count(), 3)), [0, 1, 2], None),
        ("empty input", lambda: _first(f([], 3)), [], "don't yield an empty batch at the end"),
        ("exact multiple", lambda: _first(f([1, 2, 3, 4], 2)), [[1, 2], [3, 4]], "don't yield an empty batch at the end"),
        ("size 0", lambda: _first(f([1, 2], 0)), Raises(ValueError), None),
    ])


def check_sliding_window(f):
    _scenarios("sliding_window", [
        ("basic", lambda: list(f([1, 2, 3, 4], 2)), [(1, 2), (2, 3), (3, 4)], "yield tuples"),
        ("window of 3", lambda: list(f("abcd", 3)), [("a", "b", "c"), ("b", "c", "d")], None),
        ("window bigger than the input", lambda: list(f([1, 2], 3)), [], None),
        ("window of 1", lambda: list(f([1, 2], 1)), [(1,), (2,)], None),
        ("works on a generator", lambda: list(f((x * 10 for x in range(4)), 2)), [(0, 10), (10, 20), (20, 30)], None),
        ("lazy on an infinite iterator", lambda: list(itertools.islice(f(itertools.count(), 2), 3)), [(0, 1), (1, 2), (2, 3)], None),
        ("n = 0", lambda: list(f([1, 2], 0)), Raises(ValueError), None),
    ])


def check_parse_scores(f):
    lines = ["alice,0.9", "", "# comment line", "  bob , 0.75 ", "broken line", "carol,abc", "dave,1", "eve,0.5,extra"]
    _scenarios("parse_scores", [
        ("is a generator", lambda: inspect.isgenerator(f([])), True, None),
        ("parses, cleans and skips bad lines", lambda: list(f(lines)),
         [("alice", 0.9), ("bob", 0.75), ("dave", 1.0)],
         "skip blank lines, '#' comments, and lines that don't have exactly a name and a number"),
        ("works lazily on a generator", lambda: next(f(line for line in ["x,1", "y,2"])), ("x", 1.0), None),
    ])


def check_retry(f):
    def flaky(fail_times, exc=ConnectionError):
        state = {"calls": 0}

        def call(x):
            state["calls"] += 1
            if state["calls"] <= fail_times:
                raise exc("temporary failure")
            return x * 2
        return call, state

    def succeeds():
        fn, state = flaky(2)
        wrapped = f(times=3)(fn)
        return (wrapped(5), state["calls"])

    def gives_up():
        fn, _ = flaky(5)
        f(times=3)(fn)(1)

    def gives_up_count():
        fn, state = flaky(5)
        try:
            f(times=3)(fn)(1)
        except ConnectionError:
            pass
        return state["calls"]

    def other_errors():
        fn, state = flaky(1, exc=ValueError)
        try:
            f(times=3, exceptions=(ConnectionError,))(fn)(1)
        except ValueError:
            pass
        return state["calls"]

    def keeps_name():
        @f(times=2)
        def download():
            """Fetch a file."""
        return (download.__name__, download.__doc__)

    _scenarios("retry", [
        ("succeeds after 2 failures", succeeds, (10, 3), None),
        ("raises the last error after `times` attempts", gives_up, Raises(ConnectionError), "re-raise the error after the last attempt"),
        ("makes exactly `times` attempts", gives_up_count, 3, None),
        ("doesn't retry other exception types", other_errors, 1, "only catch the exception types in `exceptions`"),
        ("keeps the name and docstring", keeps_name, ("download", "Fetch a file."), "use @functools.wraps"),
        ("times=0 is invalid", lambda: f(times=0), Raises(ValueError), "check `times` when retry() is called, before decorating anything"),
    ])


def check_memoize(f):
    def caches():
        calls = []

        @f
        def square(x):
            calls.append(x)
            return x * x
        results = [square(3), square(3), square(4), square(3)]
        return (results, calls)

    def recursive():
        calls = [0]

        @f
        def fib(n):
            calls[0] += 1
            return n if n < 2 else fib(n - 1) + fib(n - 2)
        return (fib(25), calls[0])

    def exposes_cache():
        @f
        def add(a, b):
            return a + b
        add(1, 2)
        add(3, 4)
        return add.cache

    def keeps_name():
        @f
        def embed(word):
            """Look up an embedding."""
        return embed.__name__

    _scenarios("memoize", [
        ("computes each argument only once", caches, ([9, 9, 16, 9], [3, 4]),
         "create the cache once, outside wrapper, so it survives between calls"),
        ("speeds up recursion", recursive, (75025, 26), "the recursive calls go through the decorated name, so they're cached too"),
        ("exposes the cache as a dict keyed by the args tuple", exposes_cache, {(1, 2): 3, (3, 4): 7}, None),
        ("keeps the name", keeps_name, "embed", "use @functools.wraps"),
    ])


def check_Registry(C):
    class Linear:
        def __init__(self, w=1.0):
            self.w = w

    def setup():
        r = C()

        @r.register("linear")
        class Lin(Linear):
            pass

        @r.register("relu")
        def relu(x):
            return max(0.0, x)
        return r, Lin, relu

    def returns_unchanged():
        r = C()

        def fn(x):
            return x + 1
        return r.register("inc")(fn) is fn

    def duplicate():
        r = C()
        r.register("a")(lambda: 1)
        r.register("a")(lambda: 2)

    _scenarios("Registry", [
        ("register returns the object unchanged", returns_unchanged, True, "the inner decorator should `return obj`"),
        ("membership", lambda: ("linear" in setup()[0], "conv" in setup()[0]), (True, False), "implement __contains__"),
        ("get", lambda: setup()[0].get("relu")(-3.0), 0.0, None),
        ("build calls the object with keyword arguments", lambda: setup()[0].build("linear", w=2.5).w, 2.5, None),
        ("names() is sorted", lambda: setup()[0].names(), ["linear", "relu"], None),
        ("len", lambda: len(setup()[0]), 2, None),
        ("registering a name twice", duplicate, Raises(ValueError), None),
        ("building an unknown name", lambda: setup()[0].build("conv"), Raises(KeyError), None),
    ])


def check_interleave(f):
    _scenarios("interleave", [
        ("two iterables", lambda: list(f([1, 2, 3], "ab")), [1, "a", 2, "b", 3],
         "zip() stops at the shortest input, so it loses the 3"),
        ("three iterables", lambda: list(f("ad", "be", "cf")), ["a", "b", "c", "d", "e", "f"], None),
        ("uneven lengths", lambda: list(f([1], [2, 3, 4], [5, 6])), [1, 2, 5, 3, 6, 4],
         "when one runs out, keep going with the others"),
        ("no iterables", lambda: list(f()), [], None),
        ("an empty one", lambda: list(f([], [1, 2])), [1, 2], None),
        ("lazy with an infinite iterator", lambda: list(itertools.islice(f(itertools.count(), "xy"), 6)),
         [0, "x", 1, "y", 2, 3], "don't convert the inputs to lists"),
    ])


def check_temporary_attrs(f):
    class Model:
        pass

    def make():
        m = Model()
        m.training = True
        m.dropout = 0.5
        return m

    def inside_and_after():
        m = make()
        with f(m, training=False, dropout=0.0):
            inside = (m.training, m.dropout)
        return (inside, (m.training, m.dropout))

    def new_attribute():
        m = make()
        with f(m, debug=True):
            inside = m.debug
        return (inside, hasattr(m, "debug"))

    def restores_on_error():
        m = make()
        try:
            with f(m, training=False):
                raise RuntimeError("boom")
        except RuntimeError:
            pass
        return m.training

    def propagates():
        with f(make(), training=False):
            raise RuntimeError("boom")

    def nested():
        m = make()
        with f(m, training=False):
            with f(m, training="eval"):
                pass
            middle = m.training
        return (middle, m.training)

    _scenarios("temporary_attrs", [
        ("sets attributes inside, restores them after", inside_and_after, ((False, 0.0), (True, 0.5)), None),
        ("a new attribute is removed afterwards", new_attribute, (True, False),
         "if the attribute didn't exist before, delete it on exit (delattr)"),
        ("restores even if the block raises", restores_on_error, True, "use try/finally around the yield"),
        ("doesn't swallow the exception", propagates, Raises(RuntimeError), None),
        ("nesting works", nested, (False, True), None),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `take`

In [2]:
import itertools


def take(n: int, iterable) -> list:
    return list(itertools.islice(iterable, n))

In [3]:
check_take(take)

PASS  all 5 tests passed for take


`islice` does all the work: it stops after `n` items without touching the rest, and it handles inputs shorter than `n` gracefully.

A hand-written version is a good exercise in how iterators work:

```python
def take(n, iterable):
    result = []
    if n <= 0:
        return result
    for item in iterable:
        result.append(item)
        if len(result) == n:
            break
    return result
```

The order of the steps matters. Checking `len(result) == n` right after appending, and breaking straight away, means we never ask the iterator for an item we won't use. With an iterator, an item you've pulled out is gone for good, which is what the "doesn't consume more than n items" check tests.

The same trap exists with `zip`. `zip(it, range(n))` pulls one extra item out of `it`, because `zip` asks `it` for its next item before noticing that `range` has run out. `zip(range(n), it)` happens to avoid this, but only because of the argument order, which is an easy thing to break by accident.

### Exercise 2: `running_total`

In [4]:
def running_total(numbers):
    total = 0
    for x in numbers:
        total += x
        yield total

In [5]:
check_running_total(running_total)

PASS  all 5 tests passed for running_total


The local variable `total` survives between `yield`s, which is exactly what makes generators convenient: no class, no attribute, no manual state management.

`itertools.accumulate(numbers)` does the same thing, as section 4 of the lesson showed. Writing it yourself once makes it clear there's nothing magic in there.

This is the pattern behind tracking cumulative metrics over a stream, like total tokens processed or total reward in reinforcement learning.

### Exercise 3: `count_calls`

In [6]:
import functools


def count_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

In [7]:
check_count_calls(count_calls)

PASS  all 5 tests passed for count_calls


The count is stored on `wrapper` itself, the same trick as `track_time` in the lesson. Because each call to `count_calls` creates a new `wrapper` function, each decorated function gets its own counter automatically.

`wrapper.calls = 0` has to come after the `def wrapper`, since `wrapper` doesn't exist before that. Inside `wrapper`, the name `wrapper` refers to the function object, which is fine: by the time the wrapper is *called*, it definitely exists.

A version using a closure variable also works:

```python
def count_calls(func):
    calls = 0

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        nonlocal calls
        calls += 1
        return func(*args, **kwargs)
    return wrapper
```

but then the count isn't visible from outside. `nonlocal` is needed because `calls += 1` assigns to `calls`, and without it Python would treat `calls` as a new local variable inside `wrapper`, the scope rule from Module 1.

### Exercise 4: `Timer`

In [8]:
import time


class Timer:
    def __init__(self):
        self.elapsed = None

    def __enter__(self):
        self._start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.elapsed = time.perf_counter() - self._start
        return False

In [9]:
check_Timer(Timer)

PASS  all 5 tests passed for Timer


`__exit__` runs whether the block finished normally or raised an exception, so `elapsed` is always set. Returning `False` tells Python to let any exception carry on. Returning `None` (by having no `return` at all) has the same effect, since `None` is falsy, but writing `return False` makes the intent clear.

Use `time.perf_counter()` for measuring durations, not `time.time()`. `time.time()` is the wall clock, which can jump if the system clock is adjusted. `perf_counter` is a high-resolution timer that only ever moves forward, which is what you want for benchmarks.

The private `_start` attribute is set in `__enter__` rather than `__init__`, so the same `Timer` can be used for several blocks in a row, each measured separately.

## Medium

### Exercise 5: `batched`

In [10]:
import itertools


def batched(iterable, size: int):
    if size < 1:
        raise ValueError(f"size must be at least 1, got {size}")
    it = iter(iterable)
    while True:
        batch = list(itertools.islice(it, size))
        if not batch:
            return
        yield batch

In [11]:
check_batched(batched)

PASS  all 7 tests passed for batched


The key line is `it = iter(iterable)`. Calling `islice` repeatedly on the same *iterator* continues where the last call stopped. If you passed the original list to `islice` each time, you'd get the first `size` items over and over, forever.

`return` inside a generator ends it, which the `for` loop sees as the end of the data.

One subtlety: because this is a generator function, the `ValueError` for a bad `size` isn't raised when you call `batched(data, 0)`. It's raised when you first ask for an item, since none of the body runs until then. That's why the check uses `list(batched(...))`. If you want the error immediately, split it into a normal function that validates and then returns a generator:

```python
def batched(iterable, size):
    if size < 1:
        raise ValueError(...)
    return _batched(iterable, size)    # _batched is the generator
```

This is the lazy version of the `batch` function from Module 1's exercises, and it's what a `DataLoader` does for streaming datasets.

### Exercise 6: `sliding_window`

In [12]:
from collections import deque


def sliding_window(iterable, n: int):
    if n < 1:
        raise ValueError(f"n must be at least 1, got {n}")
    window = deque(maxlen=n)
    for item in iterable:
        window.append(item)
        if len(window) == n:
            yield tuple(window)

In [13]:
check_sliding_window(sliding_window)

PASS  all 7 tests passed for sliding_window


A `deque` with `maxlen=n` drops the oldest item automatically every time a new one is appended once it's full. So the deque *is* the window, sliding along one item at a time. Each step is O(1), and memory stays at `n` items no matter how long the input is.

`tuple(window)` makes a snapshot. If we yielded the deque itself, every window we'd handed out would keep changing as the deque moved on, and `list(sliding_window(...))` would end up as several references to the same deque, all showing the final window. Yielding a mutable object that you keep modifying is a classic generator bug.

Python 3.10+ has `itertools.pairwise` for the `n = 2` case.

### Exercise 7: `parse_scores`

In [14]:
def parse_scores(lines):
    for line in lines:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = [p.strip() for p in line.split(",")]
        if len(parts) != 2:
            continue
        name, raw_score = parts
        try:
            score = float(raw_score)
        except ValueError:
            continue
        yield name, score

In [15]:
check_parse_scores(parse_scores)

PASS  all 3 tests passed for parse_scores


Each kind of bad line is handled with its own `continue`, so the structure reads top to bottom as a list of rules, and the happy path ends at the `yield`.

The `try/except` only wraps `float(raw_score)`, the one line that's expected to fail. Wrapping the whole loop body in a big `try` would also hide unrelated bugs, like a typo in a variable name.

Silently skipping bad lines is the right choice for a quick script, but in a real pipeline you'd want to know how many lines were dropped. If 40% of your data is being skipped, that's a problem you need to hear about. A common approach is to count the skips and log a summary at the end, or to `yield` the bad lines to a separate place for inspection.

### Exercise 8: `retry`

In [16]:
import functools


def retry(times: int, exceptions=(Exception,)):
    if times < 1:
        raise ValueError(f"times must be at least 1, got {times}")

    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions:
                    if attempt == times:
                        raise
        return wrapper

    return decorator

In [17]:
check_retry(retry)

PASS  all 6 tests passed for retry


Three layers, each with one job: `retry` takes the settings and validates them, `decorator` takes the function, and `wrapper` does the work on each call.

`except exceptions:` works with a tuple of exception types, so only those are caught. Anything else passes straight through on the first attempt. That matters: retrying a `ConnectionError` makes sense, but retrying a `TypeError` caused by a bug in your code just wastes time and hides the real error.

A bare `raise` inside an `except` block re-raises the exception currently being handled, with its original traceback, so the error message still points to where things actually went wrong.

A production version would usually wait between attempts, often doubling the wait each time ("exponential backoff") so as not to hammer a server that's already struggling. Adding `time.sleep(delay * 2 ** (attempt - 1))` before the next attempt is all it takes. Libraries like `tenacity` provide this and more.

### Exercise 9: `memoize`

In [18]:
import functools


def memoize(func):
    @functools.wraps(func)
    def wrapper(*args):
        if args not in wrapper.cache:
            wrapper.cache[args] = func(*args)
        return wrapper.cache[args]
    wrapper.cache = {}
    return wrapper

In [19]:
check_memoize(memoize)

PASS  all 4 tests passed for memoize


`*args` collects the positional arguments into a tuple, and tuples are hashable, so they can be dictionary keys directly. That's also why caching only works for hashable arguments: a list argument would make `args` contain a list, and `args in wrapper.cache` would raise a `TypeError`.

Why does recursion speed up? When `fib` calls `fib(n - 1)`, the name `fib` refers to the *decorated* function, since `@memoize` replaced it. So the recursive calls go through the cache too. Each value from 0 to 25 is computed exactly once, 26 calls in total. Without the cache it's over 240,000.

Checking `if args not in wrapper.cache` rather than using `wrapper.cache.get(args)` matters if the function can return `None`: with `.get`, a stored `None` looks the same as "not computed yet", and the function would run again every time. It's the same idea as `deep_get` in Module 2.

This cache grows forever. `functools.lru_cache(maxsize=...)` adds a size limit, which matters for long-running programs.

## Hard

### Exercise 10: `Registry`

In [20]:
class Registry:
    def __init__(self):
        self._items = {}

    def register(self, name: str):
        def decorator(obj):
            if name in self._items:
                raise ValueError(f"{name!r} is already registered")
            self._items[name] = obj
            return obj
        return decorator

    def get(self, name: str):
        if name not in self._items:
            raise KeyError(f"{name!r} is not registered. Available: {self.names()}")
        return self._items[name]

    def build(self, name: str, **kwargs):
        return self.get(name)(**kwargs)

    def names(self) -> list[str]:
        return sorted(self._items)

    def __contains__(self, name: str) -> bool:
        return name in self._items

    def __len__(self) -> int:
        return len(self._items)

In [21]:
check_Registry(Registry)

PASS  all 8 tests passed for Registry


`register` is a method that returns a decorator, the same "decorator with arguments" structure as `retry`, except the outer layer is a method, so the inner `decorator` can reach `self._items` through the closure.

Returning `obj` unchanged is important. If `decorator` returned `None` (by forgetting the `return`), the name `LinearModel` would be set to `None` after the class definition, which is a very confusing bug.

`build` is one line because it reuses `get`, so the helpful error message listing the available names only needs to be written once. That message is worth having: when a config file says `"model": "resnet"` and the registry only has `"resnet50"`, the error tells you exactly what's available.

The mini project uses a registry like this to plug different models into a hyperparameter search.

### Exercise 11: `interleave`

In [22]:
from collections import deque


def interleave(*iterables):
    active = deque(iter(it) for it in iterables)
    while active:
        it = active.popleft()
        try:
            item = next(it)
        except StopIteration:
            continue            # this one is finished: don't put it back
        yield item
        active.append(it)

In [23]:
check_interleave(interleave)

PASS  all 6 tests passed for interleave


The deque holds the iterators that still have items, in the order they take turns. Each round: take the iterator at the front, get one item from it, yield the item, and put the iterator at the back of the queue. An iterator that raises `StopIteration` simply isn't put back, so it drops out and the others carry on.

`iter(it)` is essential. It turns each input (a list, a string, a generator) into an iterator that remembers its position. Without it, calling `next()` on a list would fail, since lists are iterables but not iterators.

This is O(1) per item thanks to `popleft` and `append` on a deque. With a list and `pop(0)`, every step would shift all the remaining iterators along.

A version with `zip_longest` and a sentinel value also works, but it gets awkward if the data itself can contain the sentinel, and it's harder to read.

### Exercise 12: `temporary_attrs`

In [24]:
from contextlib import contextmanager

_MISSING = object()


@contextmanager
def temporary_attrs(obj, **attrs):
    saved = {name: getattr(obj, name, _MISSING) for name in attrs}
    for name, value in attrs.items():
        setattr(obj, name, value)
    try:
        yield obj
    finally:
        for name, old in saved.items():
            if old is _MISSING:
                delattr(obj, name)
            else:
                setattr(obj, name, old)

In [25]:
check_temporary_attrs(temporary_attrs)

PASS  all 5 tests passed for temporary_attrs


The tricky part is telling "the attribute didn't exist" apart from "the attribute existed and was `None`". `getattr(obj, name, None)` can't tell them apart. The solution uses a **sentinel**: `_MISSING = object()` is a unique object that can't possibly be a real attribute value, so `old is _MISSING` means exactly "there was nothing here before". This is a standard Python idiom whenever `None` is a valid value.

The old values are saved *before* anything is changed, then the `try/finally` around the `yield` guarantees the restore runs, whether the block ends normally or with an exception. Since there's no `except`, the exception carries on after the `finally` runs.

Nesting works because each `with` saves whatever values exist at the moment it starts. The inner block saves `training=False` (set by the outer block) and restores that, then the outer block restores the original `True`.

`setattr(obj, "training", False)` is the same as `obj.training = False`, but with the attribute name as a string, which is what you need when the names come from `**attrs`.

## What's next

The mini project in `04_mini_project.ipynb` builds a hyperparameter search framework. Your `Registry`, `Timer` and grid generation all come back there.